# 01 — CDC Diabetes Health Indicators (BRFSS 2015)

## Research question

When we observe a performance gap between demographic subgroups (e.g. male
vs. female) in a medical prediction model, how much of that gap is genuine
algorithmic bias versus a statistical artifact of disease **prevalence**,
**calibration**, and **case mix**?

## What this notebook does

1. Loads the CDC Diabetes Health Indicators dataset (253,680 patients).
2. Confirms the outcome column and subgroup sample sizes.
3. Runs the full reusable pipeline (`src.pipeline.run_fairness_analysis`):
   - selects from six required model families by five-fold training-only CV,
   - calibrates the selected family with five-fold isotonic fold ensembling,
   - computes **raw** subgroup fairness gaps,
   - **prevalence-adjusts** the prevalence-sensitive gaps (PPV, NPV,
     predicted-positive rate, disparate impact),
   - compares a shared decision threshold vs. thresholds chosen to
     **equalize sensitivity** across subgroups (calibration adjustment),
   - runs a **case-mix waterfall** on the raw outcome gap,
   - bootstraps 95% confidence intervals for every gap.
4. Produces the first figures (calibration curve, raw vs. adjusted gap bars).

This notebook is the template every other dataset will be run through via
the *same* `run_fairness_analysis` function, so results stay comparable.

In [ ]:
import sys, os
from pathlib import Path
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src").is_dir())
RESULTS_DIR = ROOT / "results"
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.datasets import load_cdc_diabetes
from src.pipeline import run_fairness_analysis
from src import figures as figs

pd.set_option("display.width", 120)

## 1. Load and confirm the data

A previous run hit a `KeyError` on `Diabetes_binary` -- the file we have is
the **3-class** variant (`Diabetes_012`: 0 = no diabetes, 1 = prediabetes,
2 = diabetes), not the pre-binarized variant.

`datasets.load_cdc_diabetes` handles this: it derives
`Diabetes_binary = 1` for prediabetes **or** diabetes (the standard
convention for this dataset), and drops `Diabetes_012`.

The protected attribute is `Sex` (1 = male, 0 = female), as coded in
BRFSS.

In [ ]:
d = load_cdc_diabetes()

df = d["df"]
print("Dataset:", d["name"])
print("Shape:", df.shape)
print("\nColumns:", list(df.columns))
print("\nTarget column:", d["target_col"], " | Group column:", d["group_col"])
print("\nGroup labels:", d["group_labels"])

In [ ]:
# Subgroup sizes and positive-case counts, BEFORE any modeling.
# TRIPOD+AI reporting standard: always report n and n_positive per subgroup
# before any fairness comparison.
target_col, group_col = d["target_col"], d["group_col"]

for val, label in d["group_labels"].items():
    sub = df[df[group_col] == val]
    n_pos = sub[target_col].sum()
    print(f"{label:8s}: n={len(sub):,}  positives={n_pos:,}  prevalence={n_pos/len(sub):.3f}")

Both subgroups have well over 100 positive cases (5,000+), so subgroup
estimates should be reasonably stable. Note the prevalence difference
already visible here: **men have a higher diabetes prevalence than women**
in this sample. This is exactly the kind of difference that can drive
prevalence-sensitive metric gaps (PPV, NPV, predicted-positive rate) without
the model itself being "unfair" in a recall/FNR sense.

## 2. Run the full pipeline

`run_fairness_analysis` is the single reusable function described in the
README. Group A = Male, Group B = Female, so every gap below is
**(Male) − (Female)**.

- `n_boot=1000`: 1,000 bootstrap resamples per gap for 95% CIs.
- `covariate_blocks`: used by the Step 7 case-mix waterfall (demographics →
  comorbidities → behavioral → access-to-care, added cumulatively).

In [ ]:
result = run_fairness_analysis(
    df=d["df"],
    feature_cols=d["feature_cols"],
    target_col=d["target_col"],
    group_col=d["group_col"],
    group_a_value=1,            # Male
    group_b_value=0,            # Female
    group_labels=d["group_labels"],
    covariate_blocks=d["covariate_blocks"],
    threshold="prevalence",
    test_size=0.3,
    random_state=42,
    n_boot=1000,
    verbose=True,
    # Corrected path: preprocessing is fitted on training rows only,
    # and (where a clustering identifier exists) the split, selection
    # CV, calibration folds and bootstrap are patient-grouped.
    cluster_ids=None,
    preprocess_spec=d["preprocess_spec"],
)

## 3. Reading the results

**Raw gaps (Step 3).** Men have higher prevalence (~16.8% vs ~14.9%), lower
sensitivity/recall (the model catches a smaller fraction of diabetic men
than diabetic women), and a higher Brier score (slightly worse calibration
overall for men). The `equal_opportunity_diff` (sensitivity gap) and
`fnr_diff` are the two we should watch most closely for *residual* inequity.

**Prevalence adjustment (Step 4).** PPV and predicted-positive-rate gaps
are recomputed at a common prevalence, holding each subgroup's
sensitivity/specificity fixed (Bayes' rule). Watch the `attenuation_pct`:
values near 100% mean the raw gap was almost entirely a prevalence
artifact; negative values mean the gap actually *grows* (or reverses sign)
once prevalence differences are removed -- i.e. the raw gap was *masking*
a larger underlying difference.

**Bootstrap CIs (Step 5).** A `*` flags gaps whose 95% CI excludes 0.
Sensitivity/FNR gaps being significant both before and after adjustment is
the strongest signal of residual inequity.

**Calibration adjustment (Step 6).** Compares the shared 0.5 threshold to
per-subgroup thresholds chosen to equalize sensitivity. If the
`equal_opportunity_diff` collapses to ~0 under equal-sensitivity
thresholds (by construction) but `ppv_diff`/`disparate_impact_ratio` get
*worse*, that illustrates the classic fairness trade-off: you cannot
equalize sensitivity and PPV simultaneously when prevalence differs.

**Case-mix waterfall (Step 7).** This is a separate analysis on the
*outcome itself* (not the model): a logistic regression of
`Diabetes_binary ~ Sex + covariates`, with covariate blocks added one at a
time. `group_coef_logodds` is the log-odds association between being Male
and having diabetes, after adjusting for the covariates included so far.
`pct_of_raw_gap_explained` shows how much each block changes that
coefficient relative to the "Sex only" model. A negative percentage means
the block makes the Male/Female outcome gap *larger*, not smaller --
i.e. once you control for e.g. age and income, men look relatively
*more* likely to have diabetes than the raw comparison suggested.

## 4. Figures

In [ ]:
os.makedirs(RESULTS_DIR, exist_ok=True)

fig1 = figs.plot_calibration_curve(
    result["y_test"], result["prob"], result["g_test"],
    group_labels={1: "Male", 0: "Female"},
    title="Calibration by sex (calibrated model) -- CDC Diabetes",
)
fig1.savefig(RESULTS_DIR / "01_cdc_calibration.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Raw vs. prevalence-adjusted gap, with 95% bootstrap CIs, for each
# prevalence-sensitive metric.
for key in ("ppv", "npv", "predicted_positive_rate"):
    raw = result["bootstrap_raw"][key]
    adjd = result["bootstrap_prevalence_adjusted"][key]
    gap_dict = {
        "Raw": (raw["point"], raw["ci_low"], raw["ci_high"]),
        "Prevalence-adjusted": (adjd["point"], adjd["ci_low"], adjd["ci_high"]),
    }
    fig = figs.plot_gap_bars(
        gap_dict,
        title=f"{key}: raw vs. prevalence-adjusted gap (Male - Female)",
        ylabel="Gap (Male - Female)",
    )
    fig.savefig(RESULTS_DIR / f"01_cdc_gap_{key}.png", dpi=150, bbox_inches="tight")
    plt.show()

In [ ]:
fig_wf = figs.plot_case_mix_waterfall(
    result["case_mix"],
    title="CDC Diabetes: case-mix waterfall (Male vs Female outcome gap)",
)
fig_wf.savefig(RESULTS_DIR / "01_cdc_case_mix_waterfall.png", dpi=150, bbox_inches="tight")
plt.show()

## 5. Save summary table for cross-dataset pooling

We save a tidy one-row-per-metric summary (raw gap, prevalence-adjusted
gap, attenuation, bootstrap CIs) so Step 3 of the project (pooling across
datasets) can load these without re-running the pipeline.

In [ ]:
rows = []
for key in ("ppv", "npv", "predicted_positive_rate"):
    raw = result["bootstrap_raw"][key]
    adjd = result["bootstrap_prevalence_adjusted"][key]
    pa = result["prevalence_adjustment"][key]
    rows.append({
        "dataset": "cdc_diabetes",
        "metric": key,
        "raw_gap": raw["point"],
        "raw_ci_low": raw["ci_low"],
        "raw_ci_high": raw["ci_high"],
        "prevalence_adjusted_gap": adjd["point"],
        "prevalence_adjusted_ci_low": adjd["ci_low"],
        "prevalence_adjusted_ci_high": adjd["ci_high"],
        "attenuation_pct": pa["attenuation_pct"],
    })

# Sensitivity / FNR gaps don't get a prevalence adjustment (they are
# prevalence-invariant by construction), but we still record them with CIs
# as the "residual inequity" candidates.
for key in ("sensitivity", "fnr"):
    raw = result["bootstrap_raw"][key]
    rows.append({
        "dataset": "cdc_diabetes",
        "metric": key,
        "raw_gap": raw["point"],
        "raw_ci_low": raw["ci_low"],
        "raw_ci_high": raw["ci_high"],
        "prevalence_adjusted_gap": np.nan,
        "prevalence_adjusted_ci_low": np.nan,
        "prevalence_adjusted_ci_high": np.nan,
        "attenuation_pct": np.nan,
    })

summary = pd.DataFrame(rows)
summary.to_csv(RESULTS_DIR / "01_cdc_diabetes_summary.csv", index=False)
summary